# Web server log: NASA Kennedy Space Center, tháng 7/1995

Log là nguồn **append-only, bán cấu trúc, khối lượng lớn**. Bộ này là access log thật của web
server NASA KSC: khoảng 1,9 triệu request trong ~20 MB gzip, định dạng Common Log Format:

```text
199.72.81.55 - - [01/Jul/1995:00:00:01 -0400] "GET /history/apollo/ HTTP/1.0" 200 6245
host         ident user  [timestamp kèm múi giờ]     "request line"            status bytes
```

Việc của bạn, theo đúng thứ tự một pipeline thật:

1. **parse**: đọc streaming từng dòng (không `readlines()`), dòng không khớp thì tách ra danh sách reject, không làm sập pipeline;
2. **clean**: ép kiểu, đưa timestamp về UTC, phân loại host;
3. **insights**: trả lời các câu hỏi vận hành bằng số;
4. **write_clean**: ghi Parquet vào `data/clean/nasa-http-logs/`, chạy lại không nhân đôi dòng;
5. vẽ biểu đồ, rồi làm phần trắc nghiệm trong app.

Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ
qua, test gọi thẳng các hàm bạn viết. Chấm:
`uv run pytest tracks/data-engineer/modules/00-introduction/exercises/basic/02-web-server-logs -q`

In [ ]:
# explore: tìm gốc repo (thư mục có pyproject.toml), rồi tải bộ dữ liệu nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

RAW = ensure("nasa-http-logs")
CLEAN = ROOT / "data" / "clean" / "nasa-http-logs"
print(RAW, sorted(p.name for p in RAW.iterdir()))

In [ ]:
import gzip
import re
from collections.abc import Iterable
from pathlib import Path

import numpy as np
import pandas as pd

COLUMNS = ["host", "ts", "method", "path", "protocol", "status", "bytes"]
LOCAL_TZ = "America/New_York"   # -0400 trong tháng 7 (EDT), giờ của server ở Florida

## 1. Parse từng dòng

`parse_line(line)` trả `dict` với đúng các khoá trong `COLUMNS`, hoặc `None` nếu dòng hỏng:

- `ts` giữ **nguyên chuỗi** `"01/Jul/1995:00:00:01 -0400"` (đổi kiểu hàng loạt ở bước clean nhanh hơn nhiều lần so với `strptime` từng dòng);
- `status` là `int`; `bytes` là `int`, và `-` (response không có body) thành `0`;
- request line tách theo khoảng trắng: token đầu là `method`; token cuối là `protocol` nếu bắt đầu bằng `HTTP/`,
  không thì `protocol = None` (request kiểu HTTP/0.9: `"GET /page.html"`); phần giữa là `path`;
- request rỗng hoặc chỉ có một token là dòng hỏng.

Log thật có request chứa dấu `"` bên trong (`"GET /images/" HTTP/1.0"`). Regex của bạn nên vẫn đọc được dòng đó.

In [ ]:
LOG_RE = None  # TODO: regex cho Common Log Format, 5 nhóm: host, timestamp, request, status, bytes


def parse_line(line: str) -> dict | None:
    """Một dòng Common Log Format -> dict theo COLUMNS, hoặc None nếu dòng hỏng."""
    # TODO
    raise NotImplementedError


def parse_log(lines: Iterable[str]) -> tuple[pd.DataFrame, list[str]]:
    """Đọc streaming; trả (DataFrame cột COLUMNS, danh sách dòng hỏng đã bỏ ký tự xuống dòng)."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: đọc thẳng file gzip theo dòng, không giải nén ra đĩa
with gzip.open(RAW / "NASA_access_log_Jul95.gz", "rt", encoding="latin-1") as fh:
    raw_df, rejected = parse_log(fh)
print(f"{len(raw_df):,} request, {len(rejected)} dòng hỏng")
for line in rejected[:10]:
    print(repr(line))

## 2. Làm sạch

`clean(df)` trả bản mới (không sửa `df` gốc):

- `ts`: `datetime64` **UTC** (đọc chuỗi có offset `-0400` rồi `tz_convert("UTC")`). Lưu UTC, chỉ đổi về giờ địa phương khi phân tích;
- `status`, `bytes`: kiểu số nguyên;
- thêm cột `host_type`: `"ip"` nếu host là địa chỉ IPv4, `"local"` nếu host không có dấu chấm,
  còn lại là đuôi tên miền viết thường (`"com"`, `"edu"`, `"gov"`, `"jp"`...);
- sắp theo `ts`. **Không** xoá dòng trùng: log không có khoá, hai request giống hệt trong cùng giây là hai request thật.

In [ ]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    """ts -> UTC, ép kiểu status/bytes, thêm host_type, sắp theo ts. Không dedup."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: dữ liệu sau khi làm sạch
df = clean(raw_df)
df.info()
df.head()

## 3. Insight

`insights(df, rejected)` trả `dict` với đúng các khoá dưới. Ngày và giờ tính theo **giờ địa phương
của server** (`LOCAL_TZ`); chuỗi thời gian dạng `"YYYY-MM-DD HH:MM:SS"`, ngày dạng `"YYYY-MM-DD"`.

| Khoá | Nghĩa |
| --- | --- |
| `requests`, `rejected`, `reject_rate` | số dòng đọc được, số dòng hỏng, tỉ lệ hỏng trên tổng số dòng |
| `unique_hosts`, `total_gb` | số host khác nhau, tổng byte / 1e9 |
| `not_found`, `top_404_path`, `top_404_count` | số request 404, đường dẫn 404 nhiều nhất và số lần |
| `peak_day`, `peak_day_requests` | ngày nhiều request nhất và số request ngày đó |
| `busiest_hour`, `quietest_hour` | giờ trong ngày (0–23) có tổng request lớn nhất / nhỏ nhất |
| `weekday_avg`, `weekend_avg` | số request trung bình mỗi ngày thường / mỗi ngày cuối tuần (tính trên các ngày có trong log) |
| `last_request` | thời điểm request cuối cùng |
| `max_gap_minutes`, `max_gap_start` | khoảng lặng dài nhất giữa hai request liên tiếp (phút) và thời điểm bắt đầu |
| `top_bytes_host_type` | `host_type` tải nhiều byte nhất |

In [ ]:
def insights(df: pd.DataFrame, rejected: list[str]) -> dict:
    """Các con số trong bảng trên, tính theo giờ LOCAL_TZ."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: các con số để trả lời trắc nghiệm
INSIGHTS = insights(df, rejected)
INSIGHTS

## 4. Biểu đồ

Vẽ ít nhất hai biểu đồ (mỗi biểu đồ một cell `explore`):

1. số request **theo ngày** cả tháng: ngày nào vọt lên, vì sao? (tra lịch phóng tàu con thoi tháng 7/1995);
2. số request **theo giờ trong ngày**, tách ngày thường và cuối tuần: người dùng là ai, ở đâu?
3. (tuỳ chọn) top đường dẫn 404, hoặc byte theo `host_type`.

In [ ]:
# explore: request theo ngày
# TODO: bar chart số request mỗi ngày (giờ LOCAL_TZ), tô khác màu ngày cuối tuần

In [ ]:
# explore: nhịp trong ngày
# TODO: đường request trung bình theo giờ (0-23), một đường ngày thường, một đường cuối tuần

In [ ]:
# explore: (tuỳ chọn) top 404 hoặc byte theo host_type

## 5. Ghi dữ liệu sạch

`write_clean(df, out_dir)` ghi **một** tệp `out_dir/requests.parquet` (ghi đè, tạo thư mục nếu
chưa có) và trả đường dẫn. Ghi ra tệp tạm rồi `replace` để không bao giờ để lại tệp dở. Chạy notebook
hai lần thì `data/clean/nasa-http-logs/` vẫn đúng số dòng: pipeline **idempotent**.

In [ ]:
def write_clean(df: pd.DataFrame, out_dir: Path) -> Path:
    """Ghi đè out_dir/requests.parquet (qua tệp tạm), trả đường dẫn."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: ghi data/clean/nasa-http-logs/requests.parquet
path = write_clean(df, CLEAN)
print(path, f"{path.stat().st_size / 1e6:.1f} MB", len(pd.read_parquet(path)), "dòng")

## 6. Báo cáo bài học

Làm phần trắc nghiệm trong app sau khi chạy notebook.